# Demo of matching a feature table with an authentic compound library

The internal data structure represent a feature as a simple JSON-style dictionary. 
We convert a feature table into a list of dictionaries, and use the `match_features` function in `asari.tools` to match features between two lists. The two lists can be from two feature tables. In this demo, one is from an authentic compound library, using the same techniques. 

One can reuse this notebook for annotation using authentic compound library, or compare and match features from two tables.

SL 2026-03-18

In [1]:
import json
from asari.tools.file_io import read_features_from_asari_table
from asari.tools import match_features as mf

In [2]:
# These two files are in GitHub asari repo under test/
input_file = "../test/HighOnly_HILICpos_preferred_Feature_table.tsv"
authLib_hilicpos = "../test/v2_authlib2024_hilicpos.json"

In [4]:
# Read a feature table and convert to list of dictionaries
num_samples, list_features = read_features_from_asari_table(open(input_file).read())
len(list_features), list_features[0]

table header looks like: 
   ['id_number', 'mz', 'rtime', 'rtime_left_base', 'rtime_right_base', 'parent_masstrack_id', 'peak_area', 'cSelectivity', 'goodness_fitting', 'snr', 'detection_counts', 'batch5_HILICpos_MT_20230411_007', 'batch3_HILICpos_MT_20230407_007', 'batch2_HILICpos_MT_20230405_007', 'batch4_HILICpos_MT_20230409_007']
Read 1118 feature lines


(1118,
 {'id_number': 'F0',
  'id': 'F0',
  'mz': 67.2488,
  'rtime': 244.95,
  'apex': 244.95,
  'left_base': 242.92,
  'right_base': 246.97,
  'parent_masstrack_id': '0',
  'peak_area': 1094235029.0,
  'cSelectivity': 0.78,
  'goodness_fitting': 0.93,
  'snr': 55700.0,
  'detection_counts': 4})

In [ ]:
# load authentic compound library
authLib = json.load(open(authLib_hilicpos))
len(authLib), authLib[0]

(840,
 {'id': 'v2r2024_pos_97.0735_HILIC_469.21',
  'mz': 97.0735,
  'rtime': 469.21,
  'ion': 'M+Na[1+]',
  'isotope': 'M0',
  'name': '1,3-Diaminopropane',
  'identifier': 'HMDB0000002',
  'HMDB': 'HMDB0000002',
  'PubChem': '',
  'method': 'MT8min',
  'formula': 'C3H10N2',
  'neutral_mass': 74.08439833125401})

In [13]:
# We match here btw two lists by m/z in 5 ppm and 20 seconds in retention time
matched = mf.list_match_lcms_features(list_features, authLib,  mz_ppm=5,  rt_tolerance=20)
list(matched.items())[:3]

Of 1118 list1 features, number of uni-direction matched features is 49.


[('F104',
  ['v2r2024_pos_94.0652_HILIC_28.35', 'v2r2024_pos_94.0652_HILIC_28.35']),
 ('F114', ['v2r2024_pos_97.0649_HILIC_29.0']),
 ('F133', ['v2r2024_pos_100.0757_HILIC_41.92'])]

In [ ]:
# Make dicts for later lookup 
#
dict_f = {feature['id']: feature for feature in list_features}
dict_lib = {feature['id']: feature for feature in authLib}

In [16]:
len(authLib), len(dict_lib)

(840, 800)

**Note**
There are a few entries in the authLib with more than one compounds. 
We skip the problem here. The `match_features` module has functions for best RT match etc. to help. 
The result `matched` is a dictionary mapping list1 ID to a list of list2 IDs. 
When comparting two feature tables, the latter would be list of feature IDs from table2.

In [17]:
matched['F133'], dict_lib['v2r2024_pos_100.0757_HILIC_41.92']

(['v2r2024_pos_100.0757_HILIC_41.92'],
 {'id': 'v2r2024_pos_100.0757_HILIC_41.92',
  'mz': 100.0757,
  'rtime': 41.92,
  'ion': 'M+H[1+]',
  'isotope': 'M0',
  'name': '2-Piperidinone',
  'identifier': 'HMDB0011749',
  'HMDB': 'HMDB0011749',
  'PubChem': '',
  'method': 'MT8min',
  'formula': 'C5H9NO',
  'neutral_mass': 99.0684139141547})

In [18]:
# Write out mapped result to a tsv table. Modify columns for new projects.
header = ['id', 'mz', 'rtime', 'lib_id', 'lib_name', 'lib_mz', 'lib_rtime', 'lib_identifier']
s = '\t'.join(header) + '\n'
for feature_id, matched_list in matched.items():
    for f2 in matched_list:
        s += '\t'.join([str(x) for x in [
            feature_id, dict_f[feature_id]['mz'], dict_f[feature_id]['rtime'],
            f2, dict_lib[f2]['name'], dict_lib[f2]['mz'], dict_lib[f2]['rtime'], 
            dict_lib[f2]['identifier']
            ]]) + '\n'

with open('test_annotation_output.tsv', 'w') as f:
    f.write(s)

In [19]:
# test JSON output, which is more complete
json_output = []
for feature_id, matched_list in matched.items():
    json_output.append(
        {feature_id: {'Feature': dict_f[feature_id], 
                      'Matched_libs': [dict_lib[lib_id] for lib_id in matched_list]}}
    )
with open('test_annotation_output.json', 'w') as f:
    json.dump(json_output, f, indent=4)

## Summary

This notebook shows how to match features in two tables and output tsv and JSON results. 